# RUN on BASE kernel!

## Meshing based on already segmented mask

- postprocessing of already generated mesh
- from surface to volume mesh

In [ ]:
import numpy
import os

import matplotlib.pyplot as plt
# import skimage
import gmsh
import sys

base = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/"

folder = "VTI_shared_view/segmentations/"

mask = numpy.load(base + folder + "labels_mask.npy")

stl_file = base + folder + "mesh_repaired_surface.stl"
export_file = base + folder + "mesh_volume-form-surface"
h = 5.0

In [ ]:
# def generate_tetra_mesh_from_stl(
#         stl_file,
#         output_file,
#         element_size=2.0):

#     gmsh.initialize()
#     gmsh.model.add("model")

#     # -------------------------
#     # Import STL
#     # -------------------------
#     gmsh.merge(stl_file)
    

#     # -------------------------
#     # Create volume from surfaces
#     # -------------------------
#     surfaces = gmsh.model.getEntities(2)  # all 2D surfaces
#     if not surfaces:
#         raise RuntimeError("No surfaces found in STL.")

#     surface_tags = [s[1] for s in surfaces]

#     # Add surface loop
#     sl = gmsh.model.geo.addSurfaceLoop(surface_tags)

#     # Add volume
#     gmsh.model.geo.addVolume([sl])
#     gmsh.model.geo.synchronize()


#     # -------------------------
#     # Uniform mesh size
#     # -------------------------
#     gmsh.option.setNumber("Mesh.CharacteristicLengthMin", element_size)
#     gmsh.option.setNumber("Mesh.CharacteristicLengthMax", element_size)




#     # Quality options
#     gmsh.option.setNumber("Mesh.Algorithm3D", 4)  # Delaunay
#     gmsh.option.setNumber("Mesh.Optimize", 1)
#     gmsh.option.setNumber("Mesh.OptimizeNetgen", 1)
#     gmsh.option.setNumber("Mesh.Smoothing", 20)

#     # -------------------------
#     # Remesh surface & Generate volume mesh
#     # -------------------------
#     gmsh.model.mesh.clear()  # clear existing mesh
#     gmsh.model.mesh.generate(2)  # remesh surfaces
#     gmsh.model.mesh.generate(3)  # then volume

#     # -------------------------
#     # Write output 
#     # -------------------------
#     gmsh.write(output_file)

#     gmsh.finalize()
#     print("Tetra mesh written to:", output_file)


# generate_tetra_mesh_from_stl(stl_file, export_file + "h = "+str(h)+"_1.vtk", h)


Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/VTI_shared_view/segmentations/mesh_repaired_surface.stl'...
Info    : 31988 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/VTI_shared_view/segmentations/mesh_repaired_surface.stl'
Info    : I'm busy! Ask me that later...
Info    : I'm busy! Ask me that later...
Info    : Writing '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/VTI_shared_view/segmentations/mesh_volume-form-surfaceh = 20.0_1.vtk'...
Info    : Done writing '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/VTI_shared_view/segmentations/mesh_volume-form-surfaceh = 20.0_1.vtk'
Tetra mesh written to: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/VTI_shared_view/segmentations/mesh_volume-form-surfaceh = 20.0_1.vtk


In [ ]:
def generate_tetra_mesh_from_stl(
        stl_file,
        output_file,
        element_size=2.0,
        surface_angle=40):

    gmsh.initialize()
    gmsh.model.add("model")

    # --------------------------------------------------
    # Import STL
    # --------------------------------------------------
    gmsh.merge(stl_file)

    # --------------------------------------------------
    # Convert STL mesh → geometry
    # --------------------------------------------------
    angle = surface_angle * numpy.pi / 180.0

    gmsh.model.mesh.classifySurfaces(
        angle,
        True,   # include boundary
        True    # force parametrizable patches
    )

    gmsh.model.mesh.createGeometry()
    gmsh.model.geo.synchronize()

    # --------------------------------------------------
    # Create volume
    # --------------------------------------------------
    surfaces = gmsh.model.getEntities(2)
    if not surfaces:
        raise RuntimeError("No surfaces found after reconstruction.")

    surface_tags = [s[1] for s in surfaces]

    sl = gmsh.model.geo.addSurfaceLoop(surface_tags)
    gmsh.model.geo.addVolume([sl])
    gmsh.model.geo.synchronize()

    # --------------------------------------------------
    # Uniform mesh size (true coarsening)
    # --------------------------------------------------
    gmsh.option.setNumber("Mesh.CharacteristicLengthMin", element_size)
    gmsh.option.setNumber("Mesh.CharacteristicLengthMax", element_size)

    gmsh.option.setNumber("Mesh.CharacteristicLengthFromPoints", 0)
    gmsh.option.setNumber("Mesh.CharacteristicLengthFromCurvature", 0)
    gmsh.option.setNumber("Mesh.CharacteristicLengthExtendFromBoundary", 0)

    # --------------------------------------------------
    # Mesh settings
    # --------------------------------------------------
    gmsh.option.setNumber("Mesh.Algorithm3D", 4)
    gmsh.option.setNumber("Mesh.Optimize", 1)
    gmsh.option.setNumber("Mesh.OptimizeNetgen", 1)
    gmsh.option.setNumber("Mesh.Smoothing", 20)

    # --------------------------------------------------
    # Generate volume mesh
    # --------------------------------------------------
    gmsh.model.mesh.generate(3)

    # --------------------------------------------------
    # Write output
    # --------------------------------------------------
    gmsh.write(output_file)

    gmsh.finalize()
    print("Tetra mesh written to:", output_file)

generate_tetra_mesh_from_stl(stl_file, export_file + "h = "+str(h)+"_2.vtk", h)


Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/VTI_shared_view/segmentations/mesh_repaired_surface.stl'...
Info    : 31988 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/VTI_shared_view/segmentations/mesh_repaired_surface.stl'
Info    : Classifying surfaces (angle: 40)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 31988 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    :  - Level 1 partition with 16051 triangles split in 2 parts because parametrized triangles are too small (2.49961e-12)
Info    :  - Level 2 partition with 8036 triangles split in 2 parts because parametrized triangles are too small (1.27151e-11)
Info    :  - Level 3 partition with 4027 triangles split in 2 parts because parametrized triangles are too small (2.9298e-11)
Info    :  - Level 4 partition with 2026 t